# Part 3: PySpark Implementation & Resilience

1. Transformations and actions with narrow vs wide dependencies
2. Skew optimization using hash partitioning and salting
3. RDD lineage-based fault tolerance
4. Checkpointing for long lineage stabilization

## 0) Spark Setup

We initialize Spark and import required functions.

In [30]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    avg,
    rand,
    floor,
    concat_ws,
    split,
    sum as spark_sum,
    count as spark_count
)

spark = (
    SparkSession.builder
    .appName("Global Logistics Telemetry Platform")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setCheckpointDir("./checkpoint")

print("Spark version:", spark.version)

Spark version: 3.5.0


## 1) Ingest Historical Dataset and Trigger Actions

`read.json(...)` is a transformation. `count()` is an action that triggers execution.

In [32]:
input_path = "/home/jovyan/data/telemetry_records_10000.json"

telemetry_df = (
    spark.read
    .option("multiline", True)
    .json(input_path)
)

telemetry_df.printSchema()
print("Total records:", telemetry_df.count())

root
 |-- acceleration: double (nullable = true)
 |-- battery_efficiency: double (nullable = true)
 |-- battery_voltage: double (nullable = true)
 |-- brake_status: boolean (nullable = true)
 |-- city: string (nullable = true)
 |-- coolant_temperature: double (nullable = true)
 |-- country: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- engine_rpm: long (nullable = true)
 |-- engine_status: string (nullable = true)
 |-- engine_temperature: double (nullable = true)
 |-- fault_code: string (nullable = true)
 |-- fleet_id: string (nullable = true)
 |-- fuel_consumption: double (nullable = true)
 |-- fuel_level: double (nullable = true)
 |-- gear_position: long (nullable = true)
 |-- heading: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- oil_pressure: double (nullable = true)
 |-- signal_strength: long (nullable = true)
 

## 2) Narrow vs Wide Dependencies

Narrow dependencies (no shuffle): `filter`, `select`.
Wide dependencies (shuffle): `repartition`, `groupBy + agg`.

In [33]:
selected_df = (
    telemetry_df
    .filter(col("engine_temperature").isNotNull())
    .select("vehicle_id", "vehicle_model", "engine_temperature")
)

print("Rows after narrow transforms:", selected_df.count())

Rows after narrow transforms: 10000


## 3) Optimization: Hash Partitioning + Salting for Data Skew

If some trucks produce 1000x logs, key skew causes slow reducers.
We mitigate this using hash partitioning plus two-stage salted aggregation.

In [34]:
partitioned_df = selected_df.repartition(8, "vehicle_id")
print("Partitions after hash repartition:", partitioned_df.rdd.getNumPartitions())

NUM_SALTS = 10

salted_df = (
    partitioned_df
    .withColumn("salt", floor(rand() * NUM_SALTS))
    .withColumn("salted_key", concat_ws("|", col("vehicle_id"), col("salt")))
)

partial_df = (
    salted_df
    .groupBy("salted_key", "vehicle_model")
    .agg(
        spark_sum("engine_temperature").alias("partial_temp_sum"),
        spark_count("engine_temperature").alias("partial_temp_count")
    )
)

unsalted_df = partial_df.withColumn(
    "vehicle_id",
    split(col("salted_key"), "\\|").getItem(0)
)

model_totals_df = (
    unsalted_df
    .groupBy("vehicle_model")
    .agg(
        spark_sum("partial_temp_sum").alias("model_temp_sum"),
        spark_sum("partial_temp_count").alias("model_temp_count")
    )
)

average_temperature = model_totals_df.select(
    "vehicle_model",
    (col("model_temp_sum") / col("model_temp_count")).alias("average_engine_temperature")
)

average_temperature.orderBy("vehicle_model").show(truncate=False)

Partitions after hash repartition: 8
+-------------+--------------------------+
|vehicle_model|average_engine_temperature|
+-------------+--------------------------+
|Actros       |95.04652137468554         |
|Atego        |94.96362909672253         |
|FH16         |95.05371229698376         |
|FM           |94.81571782178214         |
|FMX          |95.47362250879236         |
|Prima        |94.88984126984126         |
|R-Series     |95.05786963434016         |
|S-Series     |95.00246153846157         |
|Signa        |94.81004115226331         |
+-------------+--------------------------+



## 4) Validate Wide Dependencies in the Physical Plan

`explain()` reveals Exchange/shuffle operators for wide stages.

In [35]:
print("Distinct models:", average_temperature.count())
print("\nPhysical plan for final aggregation:")
average_temperature.explain(mode="formatted")

Distinct models: 9

Physical plan for final aggregation:
== Physical Plan ==
AdaptiveSparkPlan (13)
+- HashAggregate (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- HashAggregate (9)
            +- Exchange (8)
               +- HashAggregate (7)
                  +- Project (6)
                     +- Project (5)
                        +- Exchange (4)
                           +- Project (3)
                              +- Filter (2)
                                 +- Scan json  (1)


(1) Scan json 
Output [3]: [engine_temperature#241, vehicle_id#258, vehicle_model#259]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/data/telemetry_records_10000.json]
PushedFilters: [IsNotNull(engine_temperature)]
ReadSchema: struct<engine_temperature:double,vehicle_id:string,vehicle_model:string>

(2) Filter
Input [3]: [engine_temperature#241, vehicle_id#258, vehicle_model#259]
Condition : isnotnull(engine_temperature#241)

(3) Project
Output [3]: [vehicle_id#258, v

## 5) Fault Tolerance: RDDs and Lineage

Spark RDDs are resilient because lineage stores how partitions are derived.
If a partition is lost, Spark recomputes only missing partitions from lineage, avoiding heavy in-memory replication of all intermediates.

In [36]:
base_rdd = telemetry_df.rdd
print("RDD lineage for telemetry_df:")
print(base_rdd.toDebugString())

RDD lineage for telemetry_df:
b'(1) MapPartitionsRDD[56] at javaToPython at NativeMethodAccessorImpl.java:0 []\n |  MapPartitionsRDD[55] at javaToPython at NativeMethodAccessorImpl.java:0 []\n |  SQLExecutionRDD[54] at javaToPython at NativeMethodAccessorImpl.java:0 []\n |  MapPartitionsRDD[53] at javaToPython at NativeMethodAccessorImpl.java:0 []\n |  FileScanRDD[52] at javaToPython at NativeMethodAccessorImpl.java:0 []'


## 6) Checkpointing Long Lineage

Iterative transformations can create very deep lineage graphs.
Checkpointing truncates lineage after materialization and improves recovery stability/time for long iterative pipelines.

In [37]:
iterative_rdd = telemetry_df.rdd
for _ in range(120):
    iterative_rdd = iterative_rdd.map(lambda row: row)

print("Long lineage created. Applying checkpoint...")
iterative_rdd.checkpoint()

# Materialize so checkpoint is persisted and lineage is truncated
_ = iterative_rdd.count()

print("Lineage after checkpoint:")
print(iterative_rdd.toDebugString())

Long lineage created. Applying checkpoint...
Lineage after checkpoint:
b'(1) PythonRDD[57] at RDD at PythonRDD.scala:53 []\n |  ReliableCheckpointRDD[59] at count at /tmp/ipykernel_334/3155845272.py:9 []'


## 7) Save Batch Output

Persist final averages for downstream analytics/reporting.

In [39]:
output_path = "output_average_temperature"

(
    average_temperature
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_path)
)

print("Saved output to:", output_path)

Saved output to: output_average_temperature


## 8) Assignment Mapping Summary

1. Transformations and actions are implemented and demonstrated with execution triggers.
2. Partitioning and salting strategy addresses severe skew during aggregation.
3. RDD lineage and recomputation model are explained with lineage output.
4. Long lineage is simulated, checkpointed, and materialized to truncate DAG depth.

In [41]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
